# Power-Quality Disturbance Classifier — Colab runner

Run the cells **top to bottom**. Click the ▶ button on each one and wait for the
little green tick before moving to the next.

---

### Read this first: you do NOT need a GPU

All four models here — Random Forest, LightGBM, SVM and the MLP — are **CPU-only
algorithms**. If you pick a GPU runtime it will sit at 0% for the whole run and
change nothing.

The reason to use Colab is **memory**. Colab gives ~13 GB of RAM; that is what
your laptop was short of when the training kept dying. Here you can run all 10
folds in one go.

So: **Runtime → Change runtime type → CPU** (or just leave it on the default).

Expect the full run to take **20–30 minutes**, almost all of it in Step 5.


## Step 1 — What hardware did Colab give us?


In [ ]:
import os, multiprocessing, subprocess

cores = multiprocessing.cpu_count()
ram_gb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9
print(f'CPU cores : {cores}')
print(f'RAM       : {ram_gb:.1f} GB')

gpu = subprocess.run('nvidia-smi -L', shell=True, capture_output=True, text=True)
if gpu.returncode == 0:
    print(f'GPU       : {gpu.stdout.strip()}')
    print('          -> attached, but NOT used by this pipeline. Harmless.')
else:
    print('GPU       : none (correct - this pipeline does not need one)')

N_JOBS = max(1, cores)          # used by every later cell
print(f'\nWill run with --n-jobs {N_JOBS}')


## Step 2 — Connect your Google Drive

A popup will ask permission. Click through it and pick your Google account.

**Before running this**, upload your whole `pq_ensemble` folder to the top level
of your Google Drive (just drag it into drive.google.com).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Step 3 — Copy the project onto Colab's own disk

We copy out of Drive and work on Colab's local disk instead. Drive is slow for
the many small reads and writes this pipeline does, and large `.npz` files can
get mangled by Drive's syncing. Results get copied back at the end.


In [ ]:
import shutil, os

SRC = '/content/drive/MyDrive/pq_ensemble'    # <-- change if you put it elsewhere
DST = '/content/pq'

assert os.path.isdir(SRC), (
    f'Cannot find {SRC}.\n'
    'Check the folder name in your Drive and edit SRC above.')

if os.path.isdir(DST):
    shutil.rmtree(DST)
shutil.copytree(SRC, DST)
os.chdir(DST)

print('Working in:', os.getcwd())
print()
need = ['pqmodel.py', 'features.py', 'build_dataset.py', 'pipeline.py',
        'verify.py', 'make_figures.py', 'audit_leakage.py']
for f in need:
    print(('  OK   ' if os.path.exists(f) else '  MISSING  ') + f)


## Step 4 — Install the toolkits

Colab already has most of these. This only fills in what's missing.


In [ ]:
!pip install -q lightgbm

import numpy, scipy, sklearn, lightgbm, matplotlib
for m in (numpy, scipy, sklearn, lightgbm, matplotlib):
    print(f'{m.__name__:<12} {m.__version__}')
print('\nNote: if these versions differ from your laptop, results may move')
print('in the 3rd or 4th decimal place. That is normal.')


## Step 5 — Build the recordings

Two choices:

* **Already built them on your laptop?** Upload your `data` folder to Drive with
  everything else, and this cell will skip straight past — it detects the files.
* **Starting fresh?** It builds all six levels (5 noisy + clean). Takes ~10 min.

Level 5 is the clean, noise-free one.


In [ ]:
import os

os.makedirs('data', exist_ok=True)

for k in range(6):
    shard = f'data/shards/shard_{k}.npz'
    if os.path.exists(shard):
        print(f'level {k}: already present, skipping')
        continue
    print(f'--- building level {k} ---')
    !python build_dataset.py --step {k} --n-base 200 --shard-dir data/shards

# the 5-level dataset (your main result)
!python build_dataset.py --merge --steps 0 1 2 3 4 --shard-dir data/shards --out data/dataset.npz

# clean only (the best-case upper bound)
!python build_dataset.py --merge --steps 5 --shard-dir data/shards --out data/dataset_clean.npz

# all six levels together
!python build_dataset.py --merge --steps 0 1 2 3 4 5 --shard-dir data/shards --out data/dataset_all6.npz


## Step 6 — Train and score (the long one, ~20 min)

This is the main result: 5 noise levels, strict waveform-level split, 10-fold
cross-validation, four models plus four ways of combining them.

The `rm` line clears any leftover progress files. **Without it the program finds
old saved answers and skips all the work**, which would make your run meaningless.

Leave the browser tab open. Colab disconnects idle sessions.


In [ ]:
!rm -f results_oof_ckpt.npz results_base_ckpt.npz results.json results_preds.npz

!python pipeline.py --data data/dataset.npz --out results.json --folds 10 --n-jobs {N_JOBS}


## Step 7 — Prove we didn't cheat

Wants `19/19 checks passed`. The important ones scramble the labels on purpose —
the score must collapse to about 0.034, which is pure guessing for 29 classes.


In [ ]:
!python verify.py --data data/dataset.npz --n-jobs {N_JOBS}


## Step 8 — Draw the figures and show them here


In [ ]:
!python make_figures.py --results results.json --prefix fig

from IPython.display import Image, display
for f in ['fig2_class_snr_heatmap.png', 'fig1_snr_degradation.png',
          'fig3_confusion.png', 'fig4_feature_importance.png']:
    print('\n' + '='*70 + f'\n{f}\n' + '='*70)
    display(Image(f))


## Step 9 — The leakage audit

Splits the data the *wrong* way on purpose, to measure how much that inflates
the score. This is the most defensible number you'll produce.


In [ ]:
!python audit_leakage.py --data data/dataset.npz --n-jobs {N_JOBS}


## Step 10 — The clean-data run (~4 min)

Noise-free — the best case. Saved under different names so your main results
aren't overwritten.


In [ ]:
!rm -f results_clean_oof_ckpt.npz results_clean_base_ckpt.npz results_clean.json results_clean_preds.npz

!python pipeline.py --data data/dataset_clean.npz --out results_clean.json --folds 10 --n-jobs {N_JOBS}
!python make_figures.py --results results_clean.json --prefix figclean

from IPython.display import Image, display
display(Image('figclean2_class_snr_heatmap.png'))


## Step 11 — Headline summary table


In [ ]:
import json, numpy as np
from sklearn.metrics import f1_score

GATED = [15, 16, 20, 21, 22, 23, 28, 29]   # the four degenerate pairs
KEEP  = sorted(set(range(1, 30)) - set(GATED))

def summarise(results_file, preds_file, title):
    R = json.load(open(results_file))
    z = np.load(preds_file)
    sel = R['selected_ensemble']
    yte, ste = z['yte'], z['ste']
    yp = z['E_' + sel].argmax(1) + 1
    print(f'\n{title}   (best combiner: {sel})')
    print(f"  {'level':<8}{'all 29':>10}{'21 fair classes':>18}")
    lv = R.get('levels') or sorted(set(ste.tolist()), reverse=True)
    for s in lv:
        m = ste == s
        k = m & ~np.isin(yte, GATED)
        nm = 'clean' if s == 999 else f'{s} dB'
        print(f'  {nm:<8}{f1_score(yte[m], yp[m], average="macro"):>10.4f}'
              f'{f1_score(yte[k], yp[k], average="macro", labels=KEEP):>18.4f}')
    k = ~np.isin(yte, GATED)
    print(f'  {"OVERALL":<8}{f1_score(yte, yp, average="macro"):>10.4f}'
          f'{f1_score(yte[k], yp[k], average="macro", labels=KEEP):>18.4f}')

summarise('results.json', 'results_preds.npz', '5 NOISE LEVELS (main result)')
try:
    summarise('results_clean.json', 'results_clean_preds.npz', 'CLEAN DATA (upper bound)')
except FileNotFoundError:
    print('\n(run Step 10 for the clean numbers)')


## Step 12 — Save everything back to Drive

**Do not skip this.** Colab wipes its local disk when the session ends. This
copies your results and figures back into Drive where they'll survive.


In [ ]:
import shutil, glob, os

OUT = '/content/drive/MyDrive/pq_ensemble_results'
os.makedirs(OUT, exist_ok=True)

saved = []
for pat in ('*.json', '*.png', 'results*_preds.npz'):
    for f in glob.glob(pat):
        shutil.copy(f, OUT)
        saved.append(os.path.basename(f))

# the datasets too, so you never have to rebuild them
os.makedirs(OUT + '/data', exist_ok=True)
for f in glob.glob('data/*.npz'):
    shutil.copy(f, OUT + '/data')
    saved.append(f)

print(f'Saved {len(saved)} files to {OUT}\n')
for f in sorted(saved):
    print('  ' + f)


---

## If something goes wrong

| What you see | What to do |
|---|---|
| `Cannot find /content/drive/MyDrive/pq_ensemble` | Your folder has a different name or sits inside another folder. Open the file browser (📁 icon, left edge), find the real path, and edit `SRC` in Step 3. |
| Step 6 finishes in seconds | The `rm` line didn't run. Run the whole Step 6 cell again from the top. |
| `^C` or the session dies mid-run | Colab disconnected. Re-run Step 6 — it resumes from where it stopped. |
| `Your session crashed after using all available RAM` | Rare here. Runtime → Restart, then re-run from Step 3. |
| Numbers differ from the laptop in the 3rd decimal | Normal — different library versions. Anything under 0.005 is agreement. |

## Would a GPU ever help?

Only if you add a **neural network on the raw signal** — recommendation #7 in
`REPORT.md`. That's the one addition where a GPU would earn its keep, and it's
also the most promising improvement left, because all four current models read
the same 191 measurements and so tend to make the same mistakes.

Everything in this notebook is CPU work. Leave the runtime on CPU.
